# 01 — Data audit

What is in the panel, where it came from, and what would have to be true for it
to be usable in a publication.

The notebook prefers the processed panel built from a verified source freeze and
falls back to the frozen Portugal reference extract, printing which one it used.
That distinction matters: the reference extract is a **transcription** from the
OECD Data Explorer, not an untouched SDMX payload, and it is adequate for
development but not for a published number.

In [1]:
import json
from pathlib import Path

import pandas as pd

PROCESSED = Path("../data/processed/panel.csv")
REFERENCE = Path("../data/reference/portugal_oecd_1995_2025.csv")

source = PROCESSED if PROCESSED.exists() else REFERENCE
panel = pd.read_csv(source)
print(f"Source in use: {source}")
print(f"Publication-grade source: {source == PROCESSED}")
panel.head()

Source in use: ..\data\reference\portugal_oecd_1995_2025.csv
Publication-grade source: False


,country,year,real_wage,productivity,wage_source,productivity_source
0,PRT,1995,34209,36.19,OECD_AV_AN_WAGE_constant_2025_PPP_USD,OECD_PDB_v2_GDPHRS_PPP_USD_per_hour_chain_link...
1,PRT,1996,35476,36.82,OECD_AV_AN_WAGE_constant_2025_PPP_USD,OECD_PDB_v2_GDPHRS_PPP_USD_per_hour_chain_link...
2,PRT,1997,36395,37.56,OECD_AV_AN_WAGE_constant_2025_PPP_USD,OECD_PDB_v2_GDPHRS_PPP_USD_per_hour_chain_link...
3,PRT,1998,36826,37.98,OECD_AV_AN_WAGE_constant_2025_PPP_USD,OECD_PDB_v2_GDPHRS_PPP_USD_per_hour_chain_link...
4,PRT,1999,38085,38.83,OECD_AV_AN_WAGE_constant_2025_PPP_USD,OECD_PDB_v2_GDPHRS_PPP_USD_per_hour_chain_link...


In [2]:
panel.groupby("country").agg(
    first_year=("year", "min"),
    last_year=("year", "max"),
    n_observations=("year", "size"),
)

,first_year,last_year,n_observations
country,,,
PRT,1995,2025,31


## Levels

Both series must be strictly positive before logging, and gaps matter more than
outliers here: a missing year breaks the growth rate on both sides of it.

In [3]:
panel[["real_wage", "productivity"]].describe()

,real_wage,productivity
count,31.000000,31.000000
mean,38417.129032,43.275161
std,2296.659063,3.675934
min,34209.000000,36.190000
25%,36736.000000,40.040000
50%,38176.000000,44.300000
75%,39012.000000,45.985000
max,44937.000000,48.700000


In [4]:
# A year gap is invisible in a describe() table but fatal to a growth series.
years = panel["year"].to_numpy()
gaps = sorted(set(range(years.min(), years.max() + 1)) - set(years.tolist()))
print(f"Missing years: {gaps if gaps else 'none'}")
print(f"Duplicated years: {int(panel['year'].duplicated().sum())}")

Missing years: none
Duplicated years: 0


## Provenance

Every tracked extract carries a provenance record: where it came from, when, and
the digest of the bytes. A number that cannot be traced to one of these is not
publishable.

In [5]:
provenance_path = REFERENCE.with_suffix(".provenance.json")
if provenance_path.exists():
    provenance = json.loads(provenance_path.read_text(encoding="utf-8"))
    for key, value in provenance.items():
        if not isinstance(value, (dict, list)):
            print(f"{key}: {value}")
else:
    print("No provenance record found next to the reference extract.")

analysis_role: hourly_productivity_specification; annual-wage/per-employed-person matched specification remains pending official API extraction
common_coverage: 1995-2025
csv_sha256: d52ed713593550b8625fc0e291d006f25b3efd71398f10860cf13d68c4d2646a
retrieved_on: 2026-08-22
source_access_mode: official OECD Data Explorer snapshot; no direct Python HTTP access in execution environment
status: frozen_reference_snapshot


## The schema guard

Canonicalisation reduces a source response to country, year and one value column,
discarding the unit, price base and observation status. `audit_series_schema`
records those attributes first, and refuses to let a series mix measurement
concepts.

The cell below demonstrates the guard on two small frames shaped like an OECD
response. They illustrate the failure mode; they are not data.

In [6]:
from wage_transmission.data.schema_audit import audit_series_schema

clean = pd.DataFrame(
    {
        "REF_AREA": ["PRT", "PRT"],
        "TIME_PERIOD": [2020, 2021],
        "OBS_VALUE": [40.0, 41.0],
        "Unit of measure": ["US dollars, PPP converted"] * 2,
        "Price base": ["Constant prices"] * 2,
        "Observation status": ["Normal value", "Provisional value"],
    }
)
schema = audit_series_schema(clean, source="DEMO", value_name="productivity")
print("units             :", schema.units)
print("price bases       :", schema.price_bases)
print("observation status:", schema.observation_statuses)
print("attributes absent :", schema.attributes_absent)

units             : ('US dollars, PPP converted',)
price bases       : ('Constant prices',)
observation status: ('Normal value', 'Provisional value')
attributes absent : ('transformation', 'revision')


In [7]:
mixed = clean.copy()
mixed.loc[0, "Price base"] = "Current prices"
try:
    audit_series_schema(mixed, source="DEMO", value_name="productivity")
except ValueError as error:
    print(f"Rejected, as it should be:\n  {error}")

Rejected, as it should be:
  DEMO mixes more than one price base: ['Constant prices', 'Current prices']. Refine the source key rather than aggregating across measurement concepts.


## What this notebook does not establish

A clean audit says the panel is internally coherent. It says nothing about
whether the wage and productivity concepts are comparable across the countries
in it, and nothing about whether the deflators align. Those are checked at the
source-freeze stage, not here.